# 07. Yurak kasalliklari xavfini bashorat qilish (Heart Disease Risk Prediction Pro Max)
### SOTA Machine Learning Pipeline, Explainable AI & Framingham Risk Analysis
**Texnologiyalar:** Python, Pandas, Scikit-learn (Ensemble Soft-Voting, Gradient Boosting, Random Forest, SVM, KNN, Stratified 5-Fold Cross-Validation, Multi-ROC, SHAP-Style XAI)

Ushbu daftarda UCI Cleveland Heart Disease ma'lumotlar to'plami asosida 5 xil model arxitekturasi o'qitilib, ularning klinik samaradorligi solishtirilgan.

## 1. Kerakli kutubxonalarni yuklash va ma'lumotlarni import qilish

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score
)

print("Barcha kutubxonalar muvaffaqiyatli yuklandi!")

In [2]:
# Datasetni yuklash va tozalash
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip().str.replace("﻿", "")
duplicates = df.duplicated().sum()
if duplicates > 0:
    df.drop_duplicates(inplace=True)
print(f"Dataset o'lchami: {df.shape[0]} ta qator, {df.shape[1]} ta ustun")
df.head()

## 2. Klinik Feature Engineering va Kardiologik Biomarkerlar
UCI Cleveland ma'lumotlar to'plamidagi xom ko'rsatkichlar asosida zamonaviy klinik indekslar hisoblanadi:
- **Duke Treadmill Score (DTS):** Mashq yuklamasi, ST depressiyasi va angina birlashmasi
- **Rate Pressure Product (RPP):** Miokard kislorod sarfi ( = rac{SBP 	imes HR}{100}$)
- **Mean Arterial Pressure (MAP):** O'rtacha arterial perfuziya bosimi
- **Chronotropic Reserve %:** Maksimal puls zahirasi yetishmovchiligi


In [3]:
# Feature Engineering
df["rpp"] = (df["trestbps"] * df["thalach"] / 100.0).round(1)
df["map"] = (df["trestbps"] * 0.7 + 25.0).round(1)
df["chronotropic_reserve"] = ((df["thalach"] / (220.0 - df["age"])) * 100.0).round(1)
est_time = (df["thalach"] - 60.0) / 12.0
df["duke_score"] = (est_time - (5.0 * df["oldpeak"]) - (4.0 * df["exang"])).round(2)

print("Yangi klinik biomarkerlar o'rtacha qiymatlari (Sog'lom vs Kasal):")
display(df.groupby("target")[["rpp", "map", "chronotropic_reserve", "duke_score"]].mean().round(2))


## 3. Korrelyatsiya Matritsasi va Bog'liqliklar Tahlili
Har bir parametrning kardiak ishemiya () bilan to'g'ridan-to'g'ri korrelyatsiyasi.

In [4]:
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("UCI Cleveland Heart Disease Korrelyatsiya Matritsasi", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Train / Test (80/20) Stratified Split va StandardScaler


In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X)

print(f"O'qitish to'plami: {X_train.shape[0]} ta namuna")
print(f"Sinov to'plami:   {X_test.shape[0]} ta namuna")

## 5. 5 ta Modelni O'qitish (KNN, SVM, RF, Gradient Boosting, SOTA Ensemble)


In [4]:
knn = KNeighborsClassifier(n_neighbors=7, weights='distance')
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42)

ensemble = VotingClassifier(
    estimators=[('knn', knn), ('svm', svm), ('rf', rf), ('gb', gb)],
    voting='soft',
    weights=[1.0, 1.2, 1.3, 1.2]
)

models = {
    'KNN (k=7)': knn,
    'SVM (RBF)': svm,
    'Random Forest': rf,
    'Gradient Boosting': gb,
    'Ensemble (Soft-Voting)': ensemble
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    prob = m.predict_proba(X_test_scaled)[:, 1]
    cv_scores = cross_val_score(m, X_all_scaled, y, cv=cv, scoring='accuracy')
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, pred) * 100, 2),
        'Recall': round(recall_score(y_test, pred) * 100, 2),
        'F1-Score': round(f1_score(y_test, pred) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, prob) * 100, 2),
        '5-Fold CV': round(cv_scores.mean() * 100, 2)
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

## 6. Multi-ROC Egri Chiziqlari (5 ta Modelning Taqqoslanishi)


In [5]:
plt.figure(figsize=(9, 6))
for name, m in models.items():
    prob = m.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc*100:.1f}%)", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Tasodifiy chegara (50%)')
plt.title('Multi-Architecture ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Xulosa va Klinik Tavsiyalar
1. **Ensemble Soft-Voting (SOTA)** eng yuqori diskriminatsion qobiliyatni (**ROC-AUC 90.69%**) va **96.97% Recall**ni qayd etdi.
2. **Gradient Boosting** aniqlik bo'yicha yetakchi bo'ldi (**86.89%**).
3. **Random Forest** 5-Fold Stratified Kross-validatsiyada eng yuqori barqarorlikni ko'rsatdi (**83.74%**).
4. Barcha parametrlar veb-platforma uchun `model_data.json` va `fallback_data.js` ga to'liq eksport qilindi.